# Bước 2. Data Collection, Understanding, Preparation: thu thập, hiểu, làm sạch và chuẩn bị dữ liệu

Môn ADY201m coi trọng việc HIỂU dữ liệu trước khi dùng. Bước này vì vậy có hai phần. Phần hiểu dữ liệu (mục 2.1 tới 2.7, 2.13, 2.14) trả lời: mỗi cột là gì, có ô trống hay dòng trùng không, giá trị có nằm trong miền hợp lý không, các cột có khớp với nhau không, hai lớp khác nhau ở đâu, và còn bao nhiêu dòng sau khi làm sạch.
Phần chuẩn bị (mục 2.8 tới 2.12, 2.15) dựng một bảng duy nhất để huấn luyện mô hình. Mỗi dòng là một địa chỉ web, có cột nhãn `phishing`, các cột đặc trưng chia theo nhóm, và các cột cho biết dòng ấy thuộc tập huấn luyện, validation hay kiểm thử ở từng cách chia. Nguyên tắc quan trọng nhất: không đặc trưng nào được chứa nhãn hay dấu vết của cách thu thập dữ liệu mà người dùng thật không có.

Bốn khái niệm dùng suốt bộ notebook:
- **Lớp dương**: địa chỉ lừa đảo (`phishing = 1`). Trong tệp gốc nhãn ngược lại, nên mục 2.1 đảo nhãn ngay khi đọc.
- **Thời điểm có được của đặc trưng**: chỉ số tương đồng do tác giả dữ liệu dựng; đặc trưng địa chỉ có ngay khi nhận được đường link; đặc trưng trang chỉ có sau khi tải trang về. Một bộ lọc tin nhắn chỉ được dùng nhóm địa chỉ.
- **Nhóm khó**: các địa chỉ chỉ là trang chủ, không có đường dẫn phía sau tên miền. Toàn bộ lớp hợp lệ nằm trong nhóm này (mục 2.6), nên đây là nơi so hai lớp công bằng nhất, giống vai trò của tuần căng thẳng trong bài mẫu.
- **Ba cách chia**: chia phân tầng theo nhãn (năm seed), chia theo tên miền đăng ký (cả một tên miền đi chung một tập) và chia theo TLD (huấn luyện trên `com`, `org`, `net`, thử trên các TLD còn lại).

Bảng dưới liệt kê bốn nhóm đặc trưng. Bước 6 sẽ ghép các nhóm thành thang bảy bộ đặc trưng để xem điểm số còn lại bao nhiêu khi bỏ từng nhóm.

| STT | Nhóm | Số cột | Ví dụ | Nguồn |
|---|---|---|---|---|
| 1 | Chỉ số tương đồng (shortcut) | 1 | `URLSimilarityIndex` | Tác giả dữ liệu dựng bằng cách so với danh sách địa chỉ hợp lệ |
| 2 | Địa chỉ (address) | 21 | `URLLength`, `NoOfSubDomain`, `IsHTTPS`, `TLDLegitimateProb` | Tác giả dữ liệu tính từ chuỗi địa chỉ |
| 3 | Trang (page) | 28 | `LineOfCode`, `NoOfImage`, `HasSocialNet` | Tác giả dữ liệu tính từ mã HTML của trang |
| 4 | Không thấy tiền tố (prefix blind) | 14 | `pb_length`, `pb_digits`, `pb_has_path` | Nhóm tự tính lại từ chuỗi địa chỉ sau khi cắt `http://`, `https://` và `www.` (mục 2.11) |

Ô dưới nạp thư viện, khai báo các hằng dùng chung (năm seed, tỷ lệ chia, ba TLD dùng để huấn luyện) và tạo thư mục `Outputs`. Ô này không in gì.

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
DATA = "../Data/"
OUT = "../Outputs/"
os.makedirs(OUT, exist_ok=True)                    # thư mục chứa sản phẩm của các bước
NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"
SEEDS = [0, 1, 2, 3, 4]                            # năm seed: mỗi seed một cách chia khác nhau
SHARES = (0.6, 0.2, 0.2)                           # tỷ lệ huấn luyện, validation, kiểm thử
SEEN_TLDS = ["com", "org", "net"]                  # ba TLD phổ biến nhất, dùng để huấn luyện trong phép thử TLD chưa thấy
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:.6g}")   # in số gọn, không dùng dạng 1e+02

## 2.1 Đọc dữ liệu và kiểm tra chất lượng
Ô dưới đọc tệp CSV, in kích thước và kiểu dữ liệu, đảo nhãn thành cột `phishing`, rồi đếm ô trống và các loại dòng trùng.
Cách đọc: số ô trống phải bằng 0. Dòng "trùng hoàn toàn" so mọi cột trừ `FILENAME`, còn dòng "URL trùng" chỉ so cột địa chỉ. Hai con số này khác nhau nghĩa là cùng một địa chỉ đã được thu thập hai lần với nội dung trang khác nhau.
Nguồn là UCI Machine Learning Repository (bộ 967, giấy phép CC BY 4.0), tệp do chính tác giả bộ dữ liệu đưa lên tháng 3/2024.

In [2]:
raw = pd.read_csv(DATA + NAME, encoding="utf-8-sig")       # utf-8-sig bỏ ký tự BOM ở đầu tệp
print("Tệp gốc:", f"{raw.shape[0]:,} dòng x {raw.shape[1]} cột")
print("Kiểu dữ liệu:", raw.dtypes.astype(str).replace({"object": "str", "string": "str"}).value_counts().to_dict())
raw["phishing"] = 1 - raw["label"]                          # trong tệp nhãn 1 là hợp lệ; từ đây 1 là lừa đảo
print("Nhãn gốc label:", raw.label.value_counts().sort_index().to_dict(), "| cột mới phishing:", raw.phishing.value_counts().sort_index().to_dict())
print("Ô trống:", int(raw.isna().sum().sum()), "| dòng trùng hoàn toàn (bỏ cột FILENAME):", int(raw.drop(columns="FILENAME").duplicated().sum()),
      "| URL trùng:", int(raw.URL.duplicated().sum()), "| FILENAME trùng:", int(raw.FILENAME.duplicated().sum()))

Tệp gốc: 235,795 dòng x 56 cột
Kiểu dữ liệu: {'int64': 41, 'float64': 10, 'str': 5}
Nhãn gốc label: {0: 100945, 1: 134850} | cột mới phishing: {0: 134850, 1: 100945}
Ô trống: 0 | dòng trùng hoàn toàn (bỏ cột FILENAME): 0 | URL trùng: 425 | FILENAME trùng: 0


**Nhận xét:** tệp có 235.795 dòng x 56 cột (41 cột số nguyên, 10 cột số thực, 5 cột chữ) và không có ô trống nào. Sau khi đảo nhãn, lớp lừa đảo có 100.945 dòng và lớp hợp lệ có 134.850 dòng, khớp với trang UCI.
Không có dòng nào trùng hoàn toàn, nhưng có 425 địa chỉ xuất hiện hai lần. Tên tệp `FILENAME` thì không trùng, tức mỗi lần xuất hiện là một lần tải trang riêng. Mục 2.3 xem kỹ các dòng trùng này.

## 2.2 Từ điển dữ liệu: mỗi cột là gì, thuộc nhóm nào, đơn vị gì
Trước khi tính bất cứ thứ gì, cần biết từng cột mang nghĩa gì. Ô dưới dựng bảng từ điển cho cả 56 cột của tệp và cột nhãn mới: tên cột, nhóm theo thời điểm có được, kiểu dữ liệu, ý nghĩa, đơn vị và số giá trị khác nhau.
Cách đọc: mỗi dòng một cột. Cột `nhom` quyết định cột ấy được đưa vào bộ đặc trưng nào ở Bước 6. Bảng được lưu vào `Outputs/table_data_dictionary.csv` để nhóm chép vào báo cáo.
**Lưu ý:** trang UCI chỉ liệt kê tên cột. Ý nghĩa ở đây được suy ra từ tên cột, từ bài gốc của Prasad và Chandra (2024) và từ các phép kiểm ở mục 2.5, nên nhóm cần đối chiếu lại với bài gốc trước khi nộp.

In [3]:
TEXT = ["FILENAME", "URL", "Domain", "TLD", "Title"]
SHORTCUT = ["URLSimilarityIndex"]
ADDRESS = ["URLLength", "DomainLength", "IsDomainIP", "CharContinuationRate", "TLDLegitimateProb", "URLCharProb", "TLDLength", "NoOfSubDomain",
           "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio", "NoOfLettersInURL", "LetterRatioInURL", "NoOfDegitsInURL", "DegitRatioInURL",
           "NoOfEqualsInURL", "NoOfQMarkInURL", "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL", "IsHTTPS"]
PAGE = [c for c in raw.columns if c not in TEXT + SHORTCUT + ADDRESS + ["label", "phishing"]]
MEANING = {
    "FILENAME": ("Tên tệp HTML đã lưu của trang", "chữ"), "URL": ("Địa chỉ web đầy đủ", "chữ"), "Domain": ("Tên miền đầy đủ, gồm cả www. nếu có", "chữ"),
    "TLD": ("Tên miền cấp cao, phần sau dấu chấm cuối của Domain", "chữ"), "Title": ("Tiêu đề của trang", "chữ"),
    "URLLength": ("Độ dài địa chỉ do tác giả tính", "ký tự"), "DomainLength": ("Độ dài tên miền", "ký tự"), "IsDomainIP": ("Tên miền là địa chỉ IP", "0 hoặc 1"),
    "URLSimilarityIndex": ("Chỉ số tương đồng với danh sách địa chỉ hợp lệ, do tác giả dựng", "0 tới 100"),
    "CharContinuationRate": ("Tỷ lệ chuỗi ký tự cùng loại liên tiếp dài nhất", "0 tới 1"),
    "TLDLegitimateProb": ("Xác suất TLD là hợp lệ, tác giả tính từ thống kê", "0 tới 1"), "URLCharProb": ("Xác suất trung bình của các ký tự, tác giả tính từ thống kê", "0 tới 1"),
    "TLDLength": ("Độ dài TLD", "ký tự"), "NoOfSubDomain": ("Số tên miền phụ (www. cũng được đếm)", "số đếm"),
    "HasObfuscation": ("Địa chỉ có ký tự bị mã hoá kiểu %xx", "0 hoặc 1"), "NoOfObfuscatedChar": ("Số ký tự bị mã hoá", "số đếm"), "ObfuscationRatio": ("Tỷ lệ ký tự bị mã hoá", "0 tới 1"),
    "NoOfLettersInURL": ("Số chữ cái trong địa chỉ do tác giả đếm", "số đếm"), "LetterRatioInURL": ("Số chữ cái chia URLLength", "0 tới 1"),
    "NoOfDegitsInURL": ("Số chữ số trong địa chỉ", "số đếm"), "DegitRatioInURL": ("Số chữ số chia URLLength", "0 tới 1"),
    "NoOfEqualsInURL": ("Số dấu =", "số đếm"), "NoOfQMarkInURL": ("Số dấu ?", "số đếm"), "NoOfAmpersandInURL": ("Số dấu &", "số đếm"),
    "NoOfOtherSpecialCharsInURL": ("Số ký tự đặc biệt khác", "số đếm"), "SpacialCharRatioInURL": ("Tỷ lệ ký tự đặc biệt", "0 tới 1"), "IsHTTPS": ("Địa chỉ dùng https", "0 hoặc 1"),
    "LineOfCode": ("Số dòng mã HTML", "dòng"), "LargestLineLength": ("Độ dài dòng mã dài nhất", "ký tự"), "HasTitle": ("Trang có tiêu đề", "0 hoặc 1"),
    "DomainTitleMatchScore": ("Mức khớp giữa tên miền và tiêu đề", "0 tới 100"), "URLTitleMatchScore": ("Mức khớp giữa địa chỉ và tiêu đề", "0 tới 100"),
    "HasFavicon": ("Trang có biểu tượng favicon", "0 hoặc 1"), "Robots": ("Trang có tệp robots.txt", "0 hoặc 1"), "IsResponsive": ("Trang co giãn theo màn hình", "0 hoặc 1"),
    "NoOfURLRedirect": ("Trang có chuyển hướng sang địa chỉ khác", "0 hoặc 1 trong tệp"), "NoOfSelfRedirect": ("Trang có chuyển hướng về chính nó", "0 hoặc 1 trong tệp"),
    "HasDescription": ("Trang có thẻ mô tả", "0 hoặc 1"), "NoOfPopup": ("Số cửa sổ bật lên", "số đếm"), "NoOfiFrame": ("Số khung iframe", "số đếm"),
    "HasExternalFormSubmit": ("Biểu mẫu gửi dữ liệu ra tên miền khác", "0 hoặc 1"), "HasSocialNet": ("Trang có liên kết mạng xã hội", "0 hoặc 1"),
    "HasSubmitButton": ("Trang có nút gửi", "0 hoặc 1"), "HasHiddenFields": ("Trang có trường ẩn", "0 hoặc 1"), "HasPasswordField": ("Trang có ô mật khẩu", "0 hoặc 1"),
    "Bank": ("Trang có từ khoá về ngân hàng", "0 hoặc 1"), "Pay": ("Trang có từ khoá về thanh toán", "0 hoặc 1"), "Crypto": ("Trang có từ khoá về tiền mã hoá", "0 hoặc 1"),
    "HasCopyrightInfo": ("Trang có dòng bản quyền", "0 hoặc 1"), "NoOfImage": ("Số ảnh", "số đếm"), "NoOfCSS": ("Số tệp CSS", "số đếm"), "NoOfJS": ("Số tệp JavaScript", "số đếm"),
    "NoOfSelfRef": ("Số liên kết về chính trang", "số đếm"), "NoOfEmptyRef": ("Số liên kết rỗng", "số đếm"), "NoOfExternalRef": ("Số liên kết ra ngoài", "số đếm"),
    "label": ("Nhãn gốc: 1 là hợp lệ, 0 là lừa đảo", "0 hoặc 1"), "phishing": ("Nhãn dùng trong đồ án: 1 là lừa đảo", "0 hoặc 1"),
}
group_of = {**{c: "văn bản" for c in TEXT}, **{c: "chỉ số tương đồng" for c in SHORTCUT}, **{c: "địa chỉ" for c in ADDRESS}, **{c: "trang" for c in PAGE}, "label": "nhãn", "phishing": "nhãn"}
dictionary = pd.DataFrame([dict(STT=k, cot=c, nhom=group_of[c], kieu=str(raw[c].dtype).replace("object", "str"), y_nghia=MEANING[c][0], don_vi=MEANING[c][1],
                                so_gia_tri_khac_nhau=raw[c].nunique()) for k, c in enumerate(raw.columns, start=1)])
dictionary.to_csv(OUT + "table_data_dictionary.csv", index=False)
print(dictionary.to_string(index=False))
print("\nSố cột theo nhóm:", dictionary.nhom.value_counts().to_dict())

 STT                        cot              nhom    kieu                                                         y_nghia             don_vi  so_gia_tri_khac_nhau
   1                   FILENAME           văn bản     str                                   Tên tệp HTML đã lưu của trang                chữ                235795
   2                        URL           văn bản     str                                              Địa chỉ web đầy đủ                chữ                235370
   3                  URLLength           địa chỉ   int64                                  Độ dài địa chỉ do tác giả tính              ký tự                   482
   4                     Domain           văn bản     str                             Tên miền đầy đủ, gồm cả www. nếu có                chữ                220086
   5               DomainLength           địa chỉ   int64                                                 Độ dài tên miền              ký tự                   101
   6                 I

**Nhận xét:** 56 cột của tệp gồm 5 cột chữ, 1 chỉ số tương đồng, 21 đặc trưng địa chỉ, 28 đặc trưng trang và cột nhãn `label`. 50 cột số dùng làm đặc trưng chính là 1 + 21 + 28. Con số "54 đặc trưng" trên trang UCI là 50 cột số này cộng 4 cột chữ `URL`, `Domain`, `TLD`, `Title` (không tính `FILENAME` và nhãn), nên bài báo phải ghi rõ cách đếm.
Ba cột của nhóm địa chỉ không phải số đo trực tiếp mà là thống kê do tác giả tính trên chính bộ dữ liệu: `TLDLegitimateProb`, `URLCharProb` và (ở nhóm riêng) `URLSimilarityIndex`. Ba cột này có thể mang thông tin về nhãn của các dòng khác, nên Bước 6 có bộ đặc trưng bỏ chúng đi.

## 2.3 Địa chỉ trùng
Ô dưới xem 425 địa chỉ xuất hiện hai lần: chúng thuộc lớp nào, có cặp nào mang hai nhãn khác nhau không, và hai lần xuất hiện có cùng mọi đặc trưng không. Sau đó ô bỏ lần xuất hiện thứ hai và đánh số lại các dòng.
Cách đọc: nếu một địa chỉ mang hai nhãn khác nhau thì phải quyết định giữ nhãn nào. Nếu hai lần xuất hiện khác nhau ở đặc trưng trang thì đó là hai lần tải trang ở hai thời điểm.

In [4]:
dup = raw[raw.URL.duplicated(keep=False)]
print("Dòng có URL xuất hiện hơn một lần:", len(dup), "| số URL khác nhau trong đó:", dup.URL.nunique())
print("Theo lớp:", dup.phishing.map({0: "legitimate", 1: "phishing"}).value_counts().to_dict())
print("URL trùng mà mang hai nhãn khác nhau:", int((dup.groupby("URL").phishing.nunique() > 1).sum()))
features_only = [c for c in raw.columns if c not in ["FILENAME", "Title"]]
print("Trong các cặp trùng URL, số cặp có cùng mọi đặc trưng số:", int(dup.duplicated(features_only, keep=False).sum() // 2), "trên", dup.URL.nunique())
df = raw.drop_duplicates("URL").reset_index(drop=True).copy()
pair_diff = dup.groupby("URL")[features_only].nunique().gt(1)                  # cột nào khác nhau giữa hai bản của cùng một địa chỉ
print("Số cặp khác nhau ở từng cột (mười cột hay khác nhất):", pair_diff.sum().sort_values(ascending=False).head(10).to_dict())
gap_pair = (dup.URLLength - dup.URL.str.len()).groupby(dup.URL).agg(lambda s: tuple(sorted(s)))
print("URLLength trừ độ dài thật, của hai bản trong mỗi cặp:", gap_pair.value_counts().to_dict())
print("Số cặp khác nhau ở ít nhất một đặc trưng trang:", int(pair_diff[PAGE].any(axis=1).sum()), "trên", dup.URL.nunique())
df["row_id"] = np.arange(len(df))
print("Sau khi bỏ URL trùng (giữ lần đầu):", f"{len(df):,}", "dòng | lớp:", df.phishing.value_counts().sort_index().to_dict())

Dòng có URL xuất hiện hơn một lần: 850 | số URL khác nhau trong đó: 425
Theo lớp: {'phishing': 850}
URL trùng mà mang hai nhãn khác nhau: 0
Trong các cặp trùng URL, số cặp có cùng mọi đặc trưng số: 0 trên 425
Số cặp khác nhau ở từng cột (mười cột hay khác nhất): {'URLLength': 425, 'LetterRatioInURL': 424, 'SpacialCharRatioInURL': 407, 'NoOfOtherSpecialCharsInURL': 289, 'DegitRatioInURL': 250, 'URLCharProb': 136, 'NoOfLettersInURL': 129, 'LargestLineLength': 72, 'LineOfCode': 37, 'NoOfExternalRef': 27}
URLLength trừ độ dài thật, của hai bản trong mỗi cặp: {(-1, 0): 425}
Số cặp khác nhau ở ít nhất một đặc trưng trang: 83 trên 425
Sau khi bỏ URL trùng (giữ lần đầu): 235,370 dòng | lớp: {0: 134850, 1: 100520}


**Nhận xét:** cả 850 dòng của 425 địa chỉ trùng đều thuộc lớp lừa đảo, không có địa chỉ nào mang hai nhãn. Không cặp nào giống nhau ở mọi đặc trưng, nhưng chỗ khác nhau chủ yếu **không** nằm ở nội dung trang. Cả 425 cặp khác nhau ở `URLLength`: trong mỗi cặp, một bản có `URLLength` ngắn hơn độ dài thật 1 ký tự, bản kia bằng đúng độ dài thật. Các cột tính từ độ dài này (`LetterRatioInURL`, `SpacialCharRatioInURL`, `DegitRatioInURL`) cũng lệch theo. Chỉ 83 trên 425 cặp khác nhau ở ít nhất một đặc trưng trang.
Nghĩa là cùng một địa chỉ đã đi qua hai cách xử lý khác nhau của tác giả: một cách cắt mất 1 ký tự khi tính độ dài, một cách thì không. Mục 2.5 sẽ cho thấy cách cắt 1 ký tự được áp cho toàn bộ lớp hợp lệ nhưng chỉ cho một phần lớp lừa đảo.
Nhóm giữ lần xuất hiện đầu và bỏ lần sau, vì nếu giữ cả hai thì một địa chỉ có thể nằm ở cả tập huấn luyện lẫn tập kiểm thử. Bảng làm việc còn 235.370 dòng: 134.850 hợp lệ và 100.520 lừa đảo (42,7%).

## 2.4 Miền giá trị của từng cột số
Ô dưới in giá trị nhỏ nhất, lớn nhất, số giá trị khác nhau, số giá trị âm và tỷ lệ giá trị bằng 0 của 50 cột số, rồi kiểm tra các cột có miền cố định (tỷ lệ trong 0 tới 1, điểm trong 0 tới 100).
Cách đọc: cột đếm hay cột độ dài không được âm, cột tỷ lệ không được vượt 1. Cột `chi_0_1` cho biết cột chỉ nhận hai giá trị. Bảng lưu ở `Outputs/table_value_ranges.csv`.

In [5]:
NUM = SHORTCUT + ADDRESS + PAGE
binary = [c for c in NUM if set(df[c].unique()) <= {0, 1}]
rows = []
for c in NUM:
    v = df[c]
    rows.append(dict(cot=c, nhom=group_of[c], nho_nhat=v.min(), lon_nhat=v.max(), so_gia_tri_khac_nhau=v.nunique(), am=int((v < 0).sum()),
                     chi_0_1=c in binary, ty_le_bang_0_pct=round(100 * (v == 0).mean(), 1)))
ranges = pd.DataFrame(rows)
ranges.to_csv(OUT + "table_value_ranges.csv", index=False)
print(ranges.to_string(index=False))
print("\nSố cột có giá trị âm:", int((ranges.am > 0).sum()), "| số cột chỉ nhận 0 hoặc 1:", len(binary), "| cột tên bắt đầu bằng NoOf nhưng chỉ nhận 0 hoặc 1:", [c for c in binary if c.startswith("NoOf")])
bounded = {c: (0, 1) for c in NUM if c.endswith(("Ratio", "RatioInURL", "Rate", "Prob"))} | {c: (0, 100) for c in ["URLSimilarityIndex", "DomainTitleMatchScore", "URLTitleMatchScore"]}
print("Cột có miền cố định nằm ngoài miền:", {c: int(((df[c] < lo) | (df[c] > hi)).sum()) for c, (lo, hi) in bounded.items() if ((df[c] < lo) | (df[c] > hi)).any()} or "không có")

                       cot              nhom   nho_nhat    lon_nhat  so_gia_tri_khac_nhau  am  chi_0_1  ty_le_bang_0_pct
        URLSimilarityIndex chỉ số tương đồng   0.155574         100                 36350   0    False                 0
                 URLLength           địa chỉ         13        6097                   482   0    False                 0
              DomainLength           địa chỉ          4         110                   101   0    False                 0
                IsDomainIP           địa chỉ          0           1                     2   0     True              99.7
      CharContinuationRate           địa chỉ          0           1                   898   0    False                 0
         TLDLegitimateProb           địa chỉ          0    0.522907                   465   0    False               0.3
               URLCharProb           địa chỉ 0.00108276   0.0908237                227293   0    False                 0
                 TLDLength      

**Nhận xét:** không cột nào có giá trị âm và không cột có miền cố định nào nằm ngoài miền. 19 cột chỉ nhận 0 hoặc 1. Hai trong số đó mang tên "số lượng" nhưng thật ra là cờ: `NoOfURLRedirect` và `NoOfSelfRedirect` chỉ nhận 0 hoặc 1, nên từ điển dữ liệu ghi đơn vị là "0 hoặc 1 trong tệp".
Nhiều cột rất lệch: `URLLength` đi từ 13 tới 6.097 ký tự, `LargestLineLength` lên tới gần 14 triệu ký tự. Đây là trang và địa chỉ thật nên không bị xoá (xem quyết định làm sạch ở mục 2.6). Mô hình tuyến tính, k-NN và MLP ở Bước 6 sẽ dùng chuẩn hoá để các cột lệch không lấn át các cột khác.

## 2.5 Các cột có khớp với nhau không
Nhiều cột địa chỉ có thể tính lại từ chính chuỗi `URL` hoặc `Domain`. Ô dưới tính lại và so với cột của tác giả, riêng cho từng lớp: độ dài tên miền, TLD, số tên miền phụ, cờ HTTPS, độ dài địa chỉ, số chữ số, số chữ cái.
Cách đọc: mỗi dòng một phép so, hai cột đầu là tỷ lệ dòng khớp ở lớp hợp lệ và lớp lừa đảo. Phép so nào khớp 100% ở một lớp mà chỉ khớp một phần ở lớp kia thì cột ấy được tính theo hai cách khác nhau, và chính sự khác nhau ấy có thể tiết lộ nhãn. Bảng lưu ở `Outputs/table_consistency_checks.csv`.

In [6]:
u = df.URL
no_scheme = u.str.replace(r"^[a-z]+://", "", regex=True)                 # bỏ http:// hoặc https://
checks = pd.DataFrame({
    "DomainLength bằng độ dài Domain": df.DomainLength == df.Domain.str.len(),
    "TLD bằng phần cuối của Domain": df.TLD.astype(str) == df.Domain.str.split(".").str[-1],
    "TLDLength bằng độ dài TLD": df.TLDLength == df.TLD.astype(str).str.len(),
    "NoOfSubDomain bằng số dấu chấm của Domain trừ 1": df.NoOfSubDomain == df.Domain.str.count(r"\.") - 1,
    "IsHTTPS khớp với tiền tố https://": df.IsHTTPS == u.str.startswith("https://").astype(int),
    "URLLength bằng độ dài URL": df.URLLength == u.str.len(),
    "URLLength bằng độ dài URL trừ 1": df.URLLength == u.str.len() - 1,
    "NoOfDegitsInURL bằng số chữ số đếm lại": df.NoOfDegitsInURL == u.str.count(r"\d"),
    "NoOfLettersInURL bằng số chữ cái đếm lại (bỏ http://)": df.NoOfLettersInURL == no_scheme.str.count(r"[a-z]"),
    "URL không có chữ in hoa": ~u.str.contains(r"[A-Z]"),
})
table = (100 * checks.groupby(df.phishing).mean()).T.rename(columns={0: "legitimate_pct", 1: "phishing_pct"})
table["toan_bo_pct"] = 100 * checks.mean()
table.round(2).to_csv(OUT + "table_consistency_checks.csv")
print(table.round(2).to_string())
gap = (no_scheme.str.count(r"[a-z]") - df.NoOfLettersInURL)
print("\nSố chữ cái đếm lại trừ NoOfLettersInURL, năm giá trị hay gặp nhất của từng lớp:")
print(gap.groupby(df.phishing.map({0: "legitimate", 1: "phishing"})).value_counts().groupby(level=0).head(5).to_string())
print("\nIsHTTPS = 1 nhưng URL bắt đầu bằng http://:", int(((df.IsHTTPS == 1) & u.str.startswith("http://")).sum()), "dòng, lớp", df.loc[(df.IsHTTPS == 1) & u.str.startswith("http://"), "phishing"].value_counts().to_dict())

phishing                                               legitimate_pct  phishing_pct  toan_bo_pct
DomainLength bằng độ dài Domain                                   100           100          100
TLD bằng phần cuối của Domain                                     100           100          100
TLDLength bằng độ dài TLD                                         100           100          100
NoOfSubDomain bằng số dấu chấm của Domain trừ 1                   100           100          100
IsHTTPS khớp với tiền tố https://                                 100         99.51        99.79
URLLength bằng độ dài URL                                           0         48.18        20.58
URLLength bằng độ dài URL trừ 1                                   100         51.82        79.42
NoOfDegitsInURL bằng số chữ số đếm lại                            100          98.8        99.49
NoOfLettersInURL bằng số chữ cái đếm lại (bỏ http://)               0         50.63        21.62
URL không có chữ in hoa       

**Nhận xét:** bốn cột khớp hoàn toàn ở cả hai lớp: `DomainLength`, `TLD`, `TLDLength` và `NoOfSubDomain`. Riêng `NoOfSubDomain` bằng số dấu chấm của tên miền trừ 1, nên `www.` cũng được đếm là một tên miền phụ: cột này cũng mang dấu vết của tiền tố.
Hai cột còn lại được tính theo hai cách khác nhau giữa hai lớp. `URLLength` ngắn hơn độ dài thật đúng 1 ký tự ở **100%** địa chỉ hợp lệ, nhưng chỉ ở 51,8% địa chỉ lừa đảo. `NoOfLettersInURL` thấp hơn số chữ cái đếm lại (sau khi bỏ `http://`) đúng 4 ở 134.846 trên 134.850 địa chỉ hợp lệ, còn ở lớp lừa đảo thì lệch 0, 1, 3, 4 hay hơn tuỳ dòng.Độ lệch này giải thích được gần trọn: tác giả không đếm 3 chữ của `www` và không đếm ký tự cuối bị cắt, nên lệch 4 ở địa chỉ có `www.` và bị cắt, lệch 3 ở địa chỉ có `www.` mà không bị cắt, lệch 1 ở địa chỉ không có `www.` mà bị cắt, lệch 0 ở địa chỉ không có cả hai. Cách giải thích này đúng với 99,997% địa chỉ hợp lệ và 99,4% địa chỉ lừa đảo. Ngoài ra có 493 địa chỉ lừa đảo bắt đầu bằng `http://` mà `IsHTTPS` vẫn bằng 1.
Nhóm **không sửa** các cột của tác giả, vì mục tiêu là đo đúng bộ dữ liệu mà các bài khác đã dùng. Thay vào đó, mục 2.11 tự tính lại một bộ đặc trưng địa chỉ từ chuỗi `URL` theo cùng một cách cho cả hai lớp.

## 2.6 Các cột suy ra từ địa chỉ và các địa chỉ cực đoan
Ô dưới tạo năm cột mới từ chuỗi `URL` và `FILENAME`: địa chỉ có bắt đầu bằng `https://www.` không, có đường dẫn sau tên miền không, có phải chỉ là trang chủ không, tiền tố chữ của tên tệp, và độ dài thật của địa chỉ. Ô cũng in năm địa chỉ dài nhất.
Cách đọc: ba cột đầu dùng cho quy tắc một dòng và cho nhóm khó ở Bước 6, **không** đưa vào bộ đặc trưng của mô hình. Cột `source_prefix` chỉ dùng để mô tả cách thu thập dữ liệu.

In [7]:
df["starts_https_www"] = u.str.startswith("https://www.").astype(int)
df["has_path"] = (u.str.replace(r"^[a-z]+://[^/]+", "", regex=True).str.len() > 0).astype(int)   # còn gì sau tên miền không
df["homepage_only"] = 1 - df.has_path                                                              # nhóm khó: chỉ là trang chủ
df["source_prefix"] = df.FILENAME.str.extract(r"^([a-z]*)")[0].replace("", "digits only")          # chỉ để mô tả, không làm đặc trưng
df["url_length_true"] = u.str.len()
summary = pd.DataFrame({
    "dia_chi": df.groupby("phishing").size(),
    "bat_dau_https_www_pct": 100 * df.groupby("phishing").starts_https_www.mean(),
    "co_duong_dan_pct": 100 * df.groupby("phishing").has_path.mean(),
    "do_dai_that_trung_vi": df.groupby("phishing").url_length_true.median(),
    "do_dai_that_lon_nhat": df.groupby("phishing").url_length_true.max(),
}).rename(index={0: "legitimate", 1: "phishing"})
print(summary.round(2).to_string())
print("\nTiền tố chữ của FILENAME theo lớp:")
print(pd.crosstab(df.source_prefix, df.phishing.map({0: "legitimate", 1: "phishing"})).to_string())
print("\nNăm địa chỉ dài nhất (độ dài thật; http viết thành hxxp để không bấm nhầm vào link lừa đảo):")
print(df.nlargest(5, "url_length_true")[["url_length_true", "phishing"]].assign(dau_url=df.URL.str.slice(0, 70).str.replace("http", "hxxp", n=1)).to_string(index=False))

            dia_chi  bat_dau_https_www_pct  co_duong_dan_pct  do_dai_that_trung_vi  do_dai_that_lon_nhat
phishing                                                                                                
legitimate   134850                    100                 0                    27                    58
phishing     100520                   2.43              59.7                    34                  6097

Tiền tố chữ của FILENAME theo lớp:
phishing       legitimate  phishing
source_prefix                      
digits only        134850     55021
m                       0       380
mw                      0     36283
op                      0       548
oph                     0      8288

Năm địa chỉ dài nhất (độ dài thật; http viết thành hxxp để không bấm nhầm vào link lừa đảo):
 url_length_true  phishing                                                                dau_url
            6097         1 hxxps://xwebhostapp.github.io/freehost//#eyj2zxjzaw9uijoimc4yljailcjjb
  

**Nhận xét:** 100% địa chỉ hợp lệ bắt đầu bằng `https://www.` và 0% có đường dẫn, tức mọi địa chỉ hợp lệ là trang chủ có dạng `https://www.<tên miền>`. Ở lớp lừa đảo chỉ 2,43% bắt đầu bằng `https://www.` và 59,7% có đường dẫn. Địa chỉ hợp lệ dài nhất chỉ có 58 ký tự, trong khi địa chỉ lừa đảo dài nhất có 6.097 ký tự.
Tên tệp cho thấy hai lớp đến từ những nguồn khác nhau: cả 134.850 tên tệp hợp lệ chỉ gồm chữ số, còn 45.499 tên tệp lừa đảo bắt đầu bằng chữ (`mw` 36.283, `oph` 8.288, `op` 548, `m` 380). Năm địa chỉ dài nhất đều là lừa đảo và chứa một chuỗi mã hoá dài hoặc nhiều ký tự `%20`.

**Quyết định làm sạch (ghi lại để viết vào mục Data của bài báo):**
- Bỏ 425 dòng có URL trùng, giữ lần xuất hiện đầu (mục 2.3).
- Không xoá và không cắt ngọn địa chỉ dài hay trang cực lớn. Đó là địa chỉ thật, và địa chỉ dài chính là một dấu hiệu của lừa đảo.
- Không sửa `URLLength`, `NoOfLettersInURL` hay `IsHTTPS` dù chúng không khớp với chuỗi địa chỉ (mục 2.5). Nhóm tính bộ đặc trưng thay thế ở mục 2.11.
- Gom 636 địa chỉ có tên miền là địa chỉ IP vào một nhóm TLD riêng `(ip address)` (mục 2.8), vì cột `TLD` của chúng chỉ là đoạn số cuối của địa chỉ IP.
- Không điền giá trị thiếu vì không có ô nào trống.

## 2.7 Thống kê mô tả theo lớp
Ô dưới tính trung bình và trung vị của 15 đặc trưng chính ở từng lớp, kèm tỷ số trung bình lừa đảo trên trung bình hợp lệ, rồi đếm tỷ lệ trang có giá trị bằng 0 ở sáu đặc trưng trang.
Cách đọc: tỷ số gần 1 nghĩa là hai lớp giống nhau ở đặc trưng ấy, tỷ số xa 1 nghĩa là đặc trưng ấy tách hai lớp. Bảng lưu ở `Outputs/table_feature_by_class.csv`.

In [8]:
KEY = ["URLSimilarityIndex", "URLLength", "DomainLength", "NoOfSubDomain", "IsHTTPS", "NoOfLettersInURL", "DegitRatioInURL", "TLDLegitimateProb",
       "LineOfCode", "NoOfImage", "NoOfJS", "NoOfExternalRef", "HasSocialNet", "HasCopyrightInfo", "HasPasswordField"]
by_class = df.groupby("phishing")[KEY].agg(["mean", "median"]).T.unstack(level=1)
by_class.columns = [f"{'legitimate' if k == 0 else 'phishing'}_{s}" for k, s in by_class.columns]
by_class["ti_so_trung_binh"] = by_class.phishing_mean / by_class.legitimate_mean.replace(0, np.nan)
by_class.round(3).to_csv(OUT + "table_feature_by_class.csv", index_label="feature")
print(by_class.round(3).to_string())
zero = pd.DataFrame({c: 100 * (df[c] == 0).groupby(df.phishing).mean() for c in ["NoOfImage", "NoOfJS", "NoOfCSS", "HasSocialNet", "HasCopyrightInfo", "HasDescription"]}).T
zero.columns = ["legitimate_bang_0_pct", "phishing_bang_0_pct"]
print("\nTỷ lệ trang có giá trị bằng 0 (trang gần như trống):")
print(zero.round(1).to_string())

                    legitimate_mean  legitimate_median  phishing_mean  phishing_median  ti_so_trung_binh
URLSimilarityIndex              100                100         49.651           51.479             0.497
URLLength                    26.229                 26         45.703               34             1.742
DomainLength                 19.229                 19         24.444               22             1.271
NoOfSubDomain                 1.162                  1          1.169                1             1.007
IsHTTPS                           1                  1          0.491                0             0.491
NoOfLettersInURL             12.933                 13         28.089               20             2.172
DegitRatioInURL               0.002                  0          0.064                0            30.226
TLDLegitimateProb             0.282              0.523          0.232            0.018             0.824
LineOfCode                  1947.49               1105 

**Nhận xét:** đặc trưng trang tách hai lớp mạnh hơn hẳn đặc trưng địa chỉ. Trang hợp lệ có trung vị 1.105 dòng mã, trong khi trang lừa đảo chỉ có 12 dòng. 83,5% trang lừa đảo không có ảnh, 77,2% không có JavaScript, 99,5% không có liên kết mạng xã hội. Ở lớp hợp lệ các tỷ lệ này chỉ từ 1% tới 20%.
Nói cách khác, phần lớn trang lừa đảo được lưu lại gần như trống, có thể vì trang đã bị gỡ hoặc chặn khi tải. Đây là lý do thứ tư khiến bài toán dễ, bên cạnh ba lối tắt ở Bước 1. Ngược lại `NoOfSubDomain` gần như bằng nhau ở hai lớp (tỷ số 1,007).Một dấu hiệu ngược đời củng cố nhận định trên: trang lừa đảo thường là trang đăng nhập giả, vậy mà chỉ 5,3% trang lừa đảo có ô mật khẩu, thấp hơn 13,8% của trang hợp lệ. Nếu trang lừa đảo được lưu khi còn hoạt động thì tỷ lệ này phải cao hơn hẳn. Lưu ý thêm: tỷ số 30,226 của `DegitRatioInURL` lớn chỉ vì trung bình của lớp hợp lệ gần bằng 0 (0,002), nên với các cột gần như toàn 0 cần đọc trung vị và tỷ lệ bằng 0 thay vì tỷ số.

## 2.8 Ghép nguồn thứ hai: bảng mã nước ISO 3166 cho TLD
Bộ dữ liệu không có cột vị trí. Nguồn thứ hai là bảng mã nước ISO 3166 (thư viện `pycountry`, cài bằng `pip install pycountry`), thêm bốn mã TLD quốc gia không có trong ISO: `uk`, `eu`, `su`, `ac`. Ô dưới gom các địa chỉ IP thành một nhóm TLD riêng, phân loại TLD thành mã nước, chung hoặc địa chỉ IP, rồi ghép tên nước theo mã TLD.
Cách đọc: phép ghép là nhiều-một, nên số dòng phải giữ nguyên 235.370 (dòng `assert` ở cuối kiểm tra điều đó). Bảng mã được lưu vào `Outputs/tld_country_codes.csv` để Bước 3 nạp vào SQL.
**Lưu ý:** TLD mã nước là nơi đăng ký tên miền, không phải nơi đặt máy chủ hay nơi người lập trang sống. Một số mã nước được dùng như TLD chung (ví dụ `co`, `io`, `me`), nên góc nhìn địa lý ở Bước 4 và 5 chỉ là mô tả, không phải bằng chứng về vị trí.

In [9]:
try:
    import pycountry                                   # bảng mã nước ISO 3166 (pip install pycountry)
except ImportError as error:
    raise ImportError("Cần cài thư viện: pip install pycountry") from error
iso = pd.DataFrame([(c.alpha_2.lower(), c.name) for c in pycountry.countries], columns=["code", "country"])
extra = pd.DataFrame([("uk", "United Kingdom"), ("eu", "European Union"), ("su", "Soviet Union (legacy)"), ("ac", "Ascension Island")], columns=["code", "country"])
iso = pd.concat([iso[~iso.code.isin(extra.code)], extra], ignore_index=True)
iso.to_csv(OUT + "tld_country_codes.csv", index=False)
tld = df.TLD.astype(str).str.lower()
df["tld_clean"] = np.where(df.IsDomainIP == 1, "(ip address)", tld)                # TLD của địa chỉ IP chỉ là đoạn số cuối, gom lại một nhóm
kind = np.where(df.tld_clean == "(ip address)", "ip address", np.where(df.tld_clean.isin(iso.code), "country code", "generic"))
df["tld_type"] = kind
df = df.merge(iso.rename(columns={"code": "tld_clean"}), on="tld_clean", how="left", validate="many_to_one")
print("Bảng mã nước:", len(iso), "mã | TLD khác nhau trước và sau khi gom địa chỉ IP:", tld.nunique(), "->", df.tld_clean.nunique())
print("Số địa chỉ theo loại TLD:", df.tld_type.value_counts().to_dict())
print("Tỷ lệ lừa đảo theo loại TLD (%):", (100 * df.groupby("tld_type").phishing.mean()).round(1).to_dict())
print("Địa chỉ có TLD mã nước tìm được tên nước:", int(df.country.notna().sum()), "trên", int((df.tld_type == "country code").sum()))
top_cc = df[df.tld_type == "country code"].groupby(["tld_clean", "country"]).agg(dia_chi=("phishing", "size"), lua_dao_pct=("phishing", lambda v: 100 * v.mean())).sort_values("dia_chi", ascending=False)
print("\nMười TLD mã nước đông nhất:")
print(top_cc.head(10).round(1).to_string())
assert len(df) == 235_370, "phép ghép làm đổi số dòng"

Bảng mã nước: 253 mã | TLD khác nhau trước và sau khi gom địa chỉ IP: 695 -> 578
Số địa chỉ theo loại TLD: {'generic': 165858, 'country code': 68876, 'ip address': 636}
Tỷ lệ lừa đảo theo loại TLD (%): {'country code': 42.0, 'generic': 42.8, 'ip address': 100.0}
Địa chỉ có TLD mã nước tìm được tên nước: 68876 trên 68876

Mười TLD mã nước đông nhất:
                                          dia_chi  lua_dao_pct
tld_clean country                                             
uk        United Kingdom                     6395            5
co        Colombia                           5408         91.5
io        British Indian Ocean Territory     4174         89.7
de        Germany                            3994         17.1
ru        Russian Federation                 3870           77
au        Australia                          2979         12.5
jp        Japan                              2219          6.2
it        Italy                              1887         11.6
fr        France   

**Nhận xét:** sau khi gom 636 địa chỉ IP thành một nhóm, còn 578 TLD khác nhau (trước đó 695, vì mỗi đoạn số cuối của địa chỉ IP bị coi là một TLD). 165.858 địa chỉ có TLD chung, 68.876 có TLD mã nước và cả 68.876 đều tìm được tên nước. Tỷ lệ lừa đảo của hai loại gần như bằng nhau (42,8% và 42,0%), còn nhóm địa chỉ IP toàn là lừa đảo.
Trong mười TLD mã nước đông nhất, `co` (91,5% lừa đảo) và `io` (89,7%) gần như toàn lừa đảo nhưng thật ra là TLD được dùng như tên chung, còn `uk` (5,0%), `jp` (6,2%) và `it` (11,6%) gần như toàn hợp lệ. Phép ghép giữ nguyên 235.370 dòng.

## 2.9 Chia dữ liệu: phân tầng, theo tên miền và theo TLD
Ô dưới tạo ba loại cột chia:
- `split_s0` tới `split_s4`: chia phân tầng theo nhãn thành 60% huấn luyện, 20% validation và 20% kiểm thử. Mỗi seed cho một cách chia khác, nên độ lệch qua năm seed ở Bước 6 phản ánh cả dao động của cách chia.
- `gsplit_s0` tới `gsplit_s4`: chia theo **tên miền đăng ký**. Mọi địa chỉ của cùng một tên miền (ví dụ cả 5.718 địa chỉ `*.web.app`) đi chung một tập, nên mô hình không thể nhớ tên miền rồi nhận lại nó ở tập kiểm thử.
- `tld_seen`: 1 nếu TLD là `com`, `org` hoặc `net`. Phép thử TLD chưa thấy huấn luyện trên các dòng này và thử trên các dòng còn lại.

Tên miền đăng ký được tính gần đúng: bỏ `www.` và số cổng, giữ hai nhãn cuối (ba nhãn với dạng `co.uk`, `com.au`). Cách đọc: bảng in mười tên miền đăng ký có nhiều địa chỉ nhất.

In [10]:
def stratified_roles(y, seed):
    """Chia phân tầng theo nhãn thành train, valid, test theo SHARES."""
    rng = np.random.default_rng(seed)
    role = np.empty(len(y), dtype=object)
    for cls in (0, 1):
        idx = np.flatnonzero(y == cls)
        rng.shuffle(idx)
        a, b = int(SHARES[0] * len(idx)), int((SHARES[0] + SHARES[1]) * len(idx))
        role[idx[:a]], role[idx[a:b]], role[idx[b:]] = "train", "valid", "test"
    return role


def registered_domain(domain):
    """Tên miền đăng ký gần đúng: bỏ www. và cổng, giữ hai nhãn cuối (ba nhãn với dạng co.uk, com.au...)."""
    host = domain.split(":")[0]
    host = host[4:] if host.startswith("www.") else host
    parts = host.split(".")
    if len(parts) >= 3 and len(parts[-1]) == 2 and parts[-2] in {"co", "com", "org", "net", "ac", "gov", "edu", "or", "ne", "go"}:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])


df["reg_domain"] = [d if ip == 1 else registered_domain(d) for d, ip in zip(df.Domain, df.IsDomainIP)]
groups = df.reg_domain.unique()
for seed in SEEDS:
    df[f"split_s{seed}"] = stratified_roles(df.phishing.to_numpy(), seed)
    rng = np.random.default_rng(100 + seed)                                    # chia theo tên miền: cả một tên miền đi chung một tập
    draw = rng.random(len(groups))
    role_of = dict(zip(groups, np.where(draw < SHARES[0], "train", np.where(draw < SHARES[0] + SHARES[1], "valid", "test"))))
    df[f"gsplit_s{seed}"] = df.reg_domain.map(role_of)
df["tld_seen"] = df.tld_clean.isin(SEEN_TLDS).astype(int)
print("Số tên miền đăng ký khác nhau:", f"{len(groups):,}", "| tên miền có từ 2 địa chỉ trở lên:", int((df.reg_domain.value_counts() > 1).sum()))
print("Mười tên miền đăng ký có nhiều địa chỉ nhất:")
print(df.groupby("reg_domain").agg(dia_chi=("phishing", "size"), lua_dao_pct=("phishing", lambda v: 100 * v.mean())).nlargest(10, "dia_chi").round(1).to_string())

Số tên miền đăng ký khác nhau: 173,977 | tên miền có từ 2 địa chỉ trở lên: 6075
Mười tên miền đăng ký có nhiều địa chỉ nhất:
                 dia_chi  lua_dao_pct
reg_domain                           
web.app             5718          100
firebaseapp.com     5546          100
repl.co             3746          100
weeblysite.com      3079          100
ipfs.io             1552          100
workers.dev         1423          100
square.site         1187          100
dweb.link            959          100
xsph.ru              928          100
pantheonsite.io      891          100


**Nhận xét:** 235.370 địa chỉ thuộc 173.977 tên miền đăng ký, trong đó 6.075 tên miền có từ hai địa chỉ trở lên. Mười tên miền đông nhất đều là dịch vụ lưu trữ miễn phí và toàn là lừa đảo: `web.app` (5.718 địa chỉ), `firebaseapp.com` (5.546), `repl.co` (3.746), `weeblysite.com` (3.079), `ipfs.io` (1.552).
Nếu chia ngẫu nhiên, mô hình chỉ cần nhớ "đuôi `web.app` là lừa đảo" là đã đúng hàng nghìn dòng ở tập kiểm thử. Cách chia theo tên miền chặn đúng lối tắt này.

## 2.10 Kiểm tra cách chia
Ô dưới in, cho seed 0, số dòng, tỷ lệ, tỷ lệ lừa đảo và tỷ lệ địa chỉ là trang chủ của từng tập ở hai cách chia. Ô cũng in tỷ lệ lừa đảo của tập kiểm thử qua năm seed, tỷ lệ địa chỉ kiểm thử có tên miền đã xuất hiện trong tập huấn luyện, và hai phần của phép thử TLD chưa thấy.
Cách đọc: chia phân tầng phải cho tỷ lệ lừa đảo giống hệt nhau ở ba tập. Chia theo tên miền không giữ được tỷ lệ ấy vì vài tên miền rất lớn rơi cả khối vào một tập, đó là cái giá phải trả. Bảng lưu ở `Outputs/table_splits.csv`.

In [11]:
rows = []
for seed in SEEDS:
    for kind in ["split", "gsplit"]:
        col = f"{kind}_s{seed}"
        for role in ["train", "valid", "test"]:
            part = df[df[col] == role]
            rows.append(dict(seed=seed, cach_chia="phân tầng" if kind == "split" else "theo tên miền", tap=role, so_dong=len(part),
                             ty_le_pct=round(100 * len(part) / len(df), 1), lua_dao_pct=round(100 * part.phishing.mean(), 2),
                             trang_chu_pct=round(100 * part.homepage_only.mean(), 1)))
split_table = pd.DataFrame(rows)
split_table.to_csv(OUT + "table_splits.csv", index=False)
print(split_table[split_table.seed == 0].to_string(index=False))
print("\nTỷ lệ lừa đảo của tập kiểm thử qua năm seed (phân tầng):", split_table.query("cach_chia == 'phân tầng' and tap == 'test'").lua_dao_pct.tolist())
print("Tỷ lệ lừa đảo của tập kiểm thử qua năm seed (theo tên miền):", split_table.query("cach_chia == 'theo tên miền' and tap == 'test'").lua_dao_pct.tolist())
overlap = []
for seed in SEEDS:
    for kind in ["split", "gsplit"]:
        col = f"{kind}_s{seed}"
        train_domains = set(df.loc[df[col] == "train", "reg_domain"])
        test = df[df[col] == "test"]
        overlap.append(dict(seed=seed, cach_chia=kind, dia_chi_kiem_thu_co_ten_mien_trong_train_pct=round(100 * test.reg_domain.isin(train_domains).mean(), 2)))
overlap = pd.DataFrame(overlap)
print("\nĐịa chỉ kiểm thử có tên miền đăng ký đã xuất hiện trong tập huấn luyện (%), trung bình năm seed:", overlap.groupby("cach_chia")["dia_chi_kiem_thu_co_ten_mien_trong_train_pct"].mean().round(2).to_dict())
seen = df.groupby("tld_seen").agg(dia_chi=("phishing", "size"), lua_dao_pct=("phishing", lambda v: 100 * v.mean()), trang_chu_pct=("homepage_only", lambda v: 100 * v.mean()))
print("\nPhép thử TLD chưa thấy (1 = com, org, net dùng để huấn luyện; 0 = các TLD còn lại chỉ dùng để thử):")
print(seen.round(2).to_string())

 seed     cach_chia   tap  so_dong  ty_le_pct  lua_dao_pct  trang_chu_pct
    0     phân tầng train   141222         60        42.71           74.4
    0     phân tầng valid    47074         20        42.71           74.5
    0     phân tầng  test    47074         20        42.71           74.8
    0 theo tên miền train   143430       60.9        43.67           72.1
    0 theo tên miền valid    48509       20.6        44.25             75
    0 theo tên miền  test    43431       18.5        37.82           81.9

Tỷ lệ lừa đảo của tập kiểm thử qua năm seed (phân tầng): [42.71, 42.71, 42.71, 42.71, 42.71]
Tỷ lệ lừa đảo của tập kiểm thử qua năm seed (theo tên miền): [37.82, 51.62, 46.65, 35.78, 43.47]

Địa chỉ kiểm thử có tên miền đăng ký đã xuất hiện trong tập huấn luyện (%), trung bình năm seed: {'gsplit': 0.0, 'split': 27.39}

Phép thử TLD chưa thấy (1 = com, org, net dùng để huấn luyện; 0 = các TLD còn lại chỉ dùng để thử):
          dia_chi  lua_dao_pct  trang_chu_pct
tld_seen      

**Nhận xét:** chia phân tầng cho đúng 141.222, 47.074 và 47.074 dòng, tỷ lệ lừa đảo 42,71% ở cả ba tập và ở cả năm seed. Nhưng trong cách chia này, trung bình **27,39%** địa chỉ kiểm thử có tên miền đăng ký đã xuất hiện trong tập huấn luyện. Chia theo tên miền đưa con số ấy về 0%.
Cái giá là tỷ lệ lừa đảo của tập kiểm thử dao động từ 35,78% tới 51,62% qua năm seed và tỷ lệ các tập lệch nhẹ khỏi 60/20/20 (seed 0: 60,9%, 20,6%, 18,5%), vì các tên miền như `web.app` rơi cả khối vào một tập. Bước 6 vì vậy báo cáo chỉ số ở ngưỡng 1% báo động giả và ROC-AUC, là những chỉ số không phụ thuộc vào tỷ lệ hai lớp.
Phép thử TLD chưa thấy có 138.220 dòng `com`, `org`, `net` để huấn luyện (35,4% lừa đảo) và 97.150 dòng để thử (53,1% lừa đảo). Hai phần khác nhau cả về tỷ lệ lừa đảo lẫn tỷ lệ trang chủ (80,7% và 65,7%), nên đây là phép thử khắt khe.

## 2.11 Đặc trưng không thấy tiền tố
Mục 2.5 và 2.6 cho thấy nhiều đặc trưng địa chỉ của tác giả mang dấu vết của tiền tố `https://www.`: `IsHTTPS`, `NoOfSubDomain` (đếm cả `www.`), `URLLength` và `NoOfLettersInURL` (tính khác nhau giữa hai lớp). Hàm `prefix_blind` dưới đây tự tính lại 14 đặc trưng địa chỉ từ chuỗi `URL` sau khi cắt `http://` hoặc `https://` và mọi `www.` ở đầu, theo cùng một cách cho cả hai lớp.
Bộ này vẫn giữ `pb_has_path` và `pb_path_length`, vì có đường dẫn hay không là thông tin người dùng thật có. Nhưng vì lớp hợp lệ không có đường dẫn nào, Bước 6 sẽ chấm bộ này thêm một lần riêng trên các địa chỉ là trang chủ, nơi hai cột ấy luôn bằng 0.
Cách đọc: bảng đầu là trung vị của từng đặc trưng ở hai lớp. Bảng sau là ví dụ: hai địa chỉ đầu chỉ khác tiền tố nên phải cho cùng một hàng số.

In [12]:
def prefix_blind(urls):
    """Đặc trưng địa chỉ tính lại từ chính URL sau khi cắt http:// hoặc https:// và www., không dùng cột nào của tác giả dữ liệu."""
    core = urls.str.replace(r"^[a-z]+://", "", regex=True).str.replace(r"^(www\.)+", "", regex=True)
    host = core.str.split("/").str[0].str.split(":").str[0]
    path = core.str.replace(r"^[^/]*", "", regex=True)                       # phần sau tên miền, ví dụ "/login?id=1"
    out = pd.DataFrame({
        "pb_length": core.str.len(),
        "pb_host_length": host.str.len(),
        "pb_path_length": path.str.len(),
        "pb_has_path": (path.str.len() > 0).astype(int),
        "pb_host_dots": host.str.count(r"\."),
        "pb_hyphens": core.str.count("-"),
        "pb_digits": core.str.count(r"\d"),
        "pb_letters": core.str.count(r"[a-z]"),
        "pb_specials": core.str.count(r"[^a-z0-9]"),
        "pb_equals": core.str.count("="),
        "pb_tld_length": host.str.split(".").str[-1].str.len(),
        "pb_host_is_ip": host.str.fullmatch(r"[0-9.]+").astype(int),
    })
    out["pb_digit_ratio"] = out.pb_digits / out.pb_length.clip(lower=1)
    out["pb_letter_ratio"] = out.pb_letters / out.pb_length.clip(lower=1)
    return out


PREFIX_BLIND = list(prefix_blind(df.URL.head(2)).columns)
df = pd.concat([df, prefix_blind(df.URL)], axis=1)
print("Số đặc trưng không thấy tiền tố:", len(PREFIX_BLIND), PREFIX_BLIND)
print(df[PREFIX_BLIND].groupby(df.phishing).median().T.rename(columns={0: "legitimate_trung_vi", 1: "phishing_trung_vi"}).round(3).to_string())
example = pd.Series(["https://www.example.com", "http://example.com", "http://www.example.com/login?id=1"])
print("\nVí dụ ba địa chỉ (hai địa chỉ đầu chỉ khác tiền tố nên phải ra cùng số):")
print(prefix_blind(example).assign(url=example).set_index("url").to_string())

Số đặc trưng không thấy tiền tố: 14 ['pb_length', 'pb_host_length', 'pb_path_length', 'pb_has_path', 'pb_host_dots', 'pb_hyphens', 'pb_digits', 'pb_letters', 'pb_specials', 'pb_equals', 'pb_tld_length', 'pb_host_is_ip', 'pb_digit_ratio', 'pb_letter_ratio']
phishing         legitimate_trung_vi  phishing_trung_vi
pb_length                         15                 26
pb_host_length                    15                 20
pb_path_length                     0                  1
pb_has_path                        0                  1
pb_host_dots                       1                  2
pb_hyphens                         0                  0
pb_digits                          0                  0
pb_letters                        14                 20
pb_specials                        1                  3
pb_equals                          0                  0
pb_tld_length                      3                  3
pb_host_is_ip                      0                  0
pb_digit_ratio 

**Nhận xét:** sau khi cắt tiền tố, địa chỉ hợp lệ có trung vị 15 ký tự và lừa đảo 26 ký tự, số chữ cái 14 so với 20, số ký tự đặc biệt 1 so với 3. Hai địa chỉ `https://www.example.com` và `http://example.com` cho cùng một hàng số, đúng như thiết kế.
Khác biệt giữa hai lớp vẫn còn, và một phần lớn nằm ở chuyện có đường dẫn hay không: trung vị của `pb_has_path` là 0 ở lớp hợp lệ và 1 ở lớp lừa đảo. Ngoài ra địa chỉ lừa đảo có nhiều dấu chấm hơn trong tên máy chủ (trung vị 2 so với 1), vì chúng hay nằm trên tên miền con của dịch vụ lưu trữ miễn phí như `abc.web.app`. Mục 2.13 đo khác biệt này bằng tương quan, cả trên toàn bộ dữ liệu lẫn riêng trên các địa chỉ là trang chủ, và Bước 6 đo bằng mô hình.

## 2.12 Tự kiểm: đặc trưng không chứa tiền tố, không chứa nhãn, các tập không chồng lên nhau
Lỗi nặng nhất của một bài phân loại là đặc trưng "nhìn trộm" nhãn hay dấu vết của cách thu thập. Ô dưới chạy năm phép kiểm, mỗi phép có dòng `assert` dừng notebook nếu sai:
1. Đổi tiền tố của **mọi** địa chỉ thành `http://`, `https://www.` rồi `https://`, tính lại 14 đặc trưng không thấy tiền tố: độ lệch lớn nhất phải bằng 0.
2. Ở mọi cách chia, mỗi dòng thuộc đúng một trong ba tập.
3. Ở cách chia theo tên miền, không tên miền đăng ký nào nằm ở cả tập huấn luyện và tập kiểm thử.
4. Không bộ đặc trưng nào chứa nhãn, tên tệp, tiền tố tên tệp, cờ `https://www.` hay chuỗi địa chỉ.
5. Không ô trống trong 64 cột đặc trưng.

Cách đọc: cả ba dòng đầu phải in độ lệch 0.0, và dòng cuối phải in "mọi phép kiểm qua".

In [13]:
# 1. Đặc trưng không thấy tiền tố phải giữ nguyên khi đổi tiền tố của mọi địa chỉ
swapped = df.URL.str.replace(r"^[a-z]+://(www\.)*", "", regex=True)
for prefix in ["http://", "https://www.", "https://"]:
    gap = (prefix_blind(prefix + swapped) - df[PREFIX_BLIND]).abs().to_numpy().max()
    print(f"Đổi mọi tiền tố thành {prefix:13s} độ lệch lớn nhất của {len(PREFIX_BLIND)} đặc trưng: {gap}")
    assert gap == 0
# 2. Mỗi dòng thuộc đúng một tập ở mọi cách chia
for seed in SEEDS:
    for kind in ["split", "gsplit"]:
        assert set(df[f"{kind}_s{seed}"].unique()) == {"train", "valid", "test"}
# 3. Chia theo tên miền: không tên miền nào nằm ở cả huấn luyện và kiểm thử
for seed in SEEDS:
    tr, te = set(df.loc[df[f"gsplit_s{seed}"] == "train", "reg_domain"]), set(df.loc[df[f"gsplit_s{seed}"] == "test", "reg_domain"])
    assert not tr & te
print("Chia theo tên miền: số tên miền chung giữa train và test ở năm seed = 0")
# 4. Không cột đặc trưng nào là nhãn hay suy ra trực tiếp từ nguồn thu thập
FEATURE_SETS = {"shortcut": SHORTCUT, "address": ADDRESS, "page": PAGE, "prefix_blind": PREFIX_BLIND}
forbidden = {"label", "phishing", "FILENAME", "source_prefix", "starts_https_www", "URL", "Title"}
assert not forbidden & set(sum(FEATURE_SETS.values(), []))
print("Các bộ đặc trưng không chứa:", sorted(forbidden))
# 5. Không ô trống trong mọi cột đặc trưng
assert df[sum(FEATURE_SETS.values(), [])].isna().sum().sum() == 0
print("Không ô trống trong", len(sum(FEATURE_SETS.values(), [])), "cột đặc trưng | mọi phép kiểm qua")

Đổi mọi tiền tố thành http://       độ lệch lớn nhất của 14 đặc trưng: 0.0
Đổi mọi tiền tố thành https://www.  độ lệch lớn nhất của 14 đặc trưng: 0.0
Đổi mọi tiền tố thành https://      độ lệch lớn nhất của 14 đặc trưng: 0.0
Chia theo tên miền: số tên miền chung giữa train và test ở năm seed = 0
Các bộ đặc trưng không chứa: ['FILENAME', 'Title', 'URL', 'label', 'phishing', 'source_prefix', 'starts_https_www']
Không ô trống trong 64 cột đặc trưng | mọi phép kiểm qua


**Nhận xét:** đổi tiền tố của cả 235.370 địa chỉ theo ba cách đều cho độ lệch 0.0, nên 14 đặc trưng không thấy tiền tố thật sự không thấy tiền tố. Ở cả năm seed của cách chia theo tên miền, không tên miền nào nằm ở cả huấn luyện lẫn kiểm thử. Bốn bộ đặc trưng (64 cột) không chứa cột nào bị cấm và không có ô trống.
Lưu ý: phép kiểm 4 chỉ bảo đảm không có cột nào **là** nhãn. Nó không bảo đảm các cột của tác giả không **chứa** dấu vết của cách thu thập, vì mục 2.5 đã cho thấy điều ngược lại. Đó là việc của thang đặc trưng ở Bước 6.

## 2.13 Thống kê mô tả của 64 đặc trưng
Trước khi đưa vào mô hình, cần nhìn thang đo và mức liên quan với nhãn của từng đặc trưng. Ô dưới in trung bình, độ lệch chuẩn, nhỏ nhất, trung vị, lớn nhất, tương quan hạng Spearman với nhãn và số giá trị khác nhau của 64 đặc trưng, rồi tính lại tương quan của bộ không thấy tiền tố chỉ trên các địa chỉ là trang chủ.
Cách đọc: tương quan dương nghĩa là giá trị càng lớn thì càng hay là lừa đảo. Tương quan chỉ nói hai thứ đi cùng nhau, chưa nói gì về nguyên nhân. Bảng lưu ở `Outputs/table_feature_statistics.csv`.

In [14]:
ALL_FEATURES = SHORTCUT + ADDRESS + PAGE + PREFIX_BLIND
stats = df[ALL_FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].rename(columns={"mean": "trung_binh", "std": "do_lech_chuan", "min": "nho_nhat", "50%": "trung_vi", "max": "lon_nhat"})
stats.insert(0, "nhom", [group_of.get(c, "không thấy tiền tố") for c in ALL_FEATURES])
stats["tuong_quan_hang_voi_nhan"] = df[ALL_FEATURES].corrwith(df.phishing, method="spearman")
stats["so_gia_tri_khac_nhau"] = df[ALL_FEATURES].nunique()
stats.round(4).to_csv(OUT + "table_feature_statistics.csv", index_label="feature")
print(stats.round(3).to_string())
strength = stats.groupby("nhom").tuong_quan_hang_voi_nhan.agg(lambda v: v.abs().max()).round(3)
print("\nTương quan hạng mạnh nhất (độ lớn) trong từng nhóm:", strength.to_dict())
home = df[df.homepage_only == 1]
varying = [c for c in PREFIX_BLIND if home[c].nunique() > 1]                 # bỏ cột hằng số trên nhóm này (ví dụ pb_has_path luôn bằng 0)
pb_home = home[varying].corrwith(home.phishing, method="spearman").abs().sort_values(ascending=False)
print("Chỉ xét", f"{len(home):,}", "địa chỉ là trang chủ (lừa đảo %.1f%%), cột hằng số bị bỏ:" % (100 * home.phishing.mean()), sorted(set(PREFIX_BLIND) - set(varying)))
print("Chỉ xét địa chỉ là trang chủ, ba đặc trưng không thấy tiền tố mạnh nhất:", pb_home.head(3).round(3).to_dict())
for name, cols in [("chỉ số tương đồng", SHORTCUT), ("địa chỉ của tác giả", ADDRESS), ("trang", PAGE)]:      # so với ba nhóm của tác giả trên cùng nhóm trang chủ
    r = home[[c for c in cols if home[c].nunique() > 1]].corrwith(home.phishing, method="spearman").abs().sort_values(ascending=False)
    print(f"Chỉ xét địa chỉ là trang chủ, nhóm {name}, mạnh nhất:", r.head(3).round(3).to_dict())
print("Trong các địa chỉ lừa đảo là trang chủ, tỷ lệ dùng https: %.1f%%" % (100 * home.loc[home.phishing == 1, "IsHTTPS"].mean()))

                                          nhom  trung_binh  do_lech_chuan  nho_nhat  trung_vi    lon_nhat  tuong_quan_hang_voi_nhan  so_gia_tri_khac_nhau
URLSimilarityIndex           chỉ số tương đồng      78.498         28.948     0.156       100         100                    -0.945                 36350
URLLength                              địa chỉ      34.546         41.333        13        27        6097                     0.397                   482
DomainLength                           địa chỉ      21.456          9.128         4        20         110                     0.213                   101
IsDomainIP                             địa chỉ       0.003          0.052         0         0           1                      0.06                     2
CharContinuationRate                   địa chỉ       0.846          0.216         0         1           1                    -0.469                   898
TLDLegitimateProb                      địa chỉ       0.261          0.252   

**Nhận xét:** tương quan mạnh nhất của từng nhóm là 0,945 ở chỉ số tương đồng, 0,842 ở nhóm trang (`NoOfImage`, `NoOfExternalRef`), 0,678 ở nhóm không thấy tiền tố (`pb_has_path`) và 0,612 ở nhóm địa chỉ (`NoOfOtherSpecialCharsInURL`). Trong nhóm địa chỉ của tác giả, `IsHTTPS` có tương quan -0,610, gần bằng đặc trưng mạnh nhất.
Khi chỉ xét 175.363 địa chỉ là trang chủ (23,1% lừa đảo), hai cột đường dẫn trở thành hằng số và bị bỏ. Đặc trưng không thấy tiền tố mạnh nhất chỉ còn tương quan 0,299 (`pb_digits`), kế đó là `pb_digit_ratio` (0,298) và `pb_letter_ratio` (0,202). Trên cùng nhóm trang chủ ấy, các cột của tác giả vẫn gần như chứa nhãn: chỉ số tương đồng có tương quan 0,977 và `IsHTTPS` có 0,963, vì chỉ 5,6% địa chỉ lừa đảo là trang chủ dùng https trong khi mọi địa chỉ hợp lệ đều dùng https. Nhóm trang vẫn còn 0,728 (`LineOfCode`). Bỏ `IsHTTPS` đi thì đặc trưng địa chỉ mạnh nhất của tác giả cũng chỉ còn 0,299 (`NoOfDegitsInURL`), bằng đúng mức của bộ không thấy tiền tố. Đây là dấu hiệu sớm rằng khi bỏ các lối tắt, phân biệt lừa đảo chỉ từ địa chỉ khó hơn nhiều so với con số 99% gợi ý.

## 2.14 Số dòng trước và sau khi làm sạch
Báo cáo phải nói rõ dữ liệu vào có bao nhiêu dòng và sau từng bước còn bao nhiêu. Ô dưới lập bảng đếm ấy và lưu vào `Outputs/table_row_counts.csv`.
Cách đọc: đọc từ trên xuống. Chỉ có một lý do xoá dòng (URL trùng). Các dòng sau không phải là xoá, mà là kích thước của các nhóm và các tập dùng ở Bước 6.

In [15]:
row_counts = pd.DataFrame([
    ("Tệp gốc PhiUSIIL_Phishing_URL_Dataset.csv", len(raw), raw.shape[1] - 1, "Dữ liệu vào (cột phishing do nhóm thêm không tính)"),
    ("Dòng bị xoá vì ô trống hoặc giá trị ngoài miền", 0, raw.shape[1] - 1, "Không có dòng nào lỗi"),
    ("Dòng bị xoá vì URL trùng", int(raw.URL.duplicated().sum()), raw.shape[1] - 1, "Giữ lần xuất hiện đầu; cả 425 dòng đều là lừa đảo"),
    ("Bảng sau khi bỏ trùng", len(df), raw.shape[1] - 1, "134.850 hợp lệ, 100.520 lừa đảo"),
    ("Địa chỉ là trang chủ (nhóm khó)", int(df.homepage_only.sum()), 1, "Toàn bộ lớp hợp lệ và một phần lớp lừa đảo"),
    ("Địa chỉ có TLD là com, org, net", int(df.tld_seen.sum()), 1, "Phần huấn luyện của phép thử TLD chưa thấy"),
    ("Địa chỉ có TLD khác", int((1 - df.tld_seen).sum()), 1, "Phần chỉ dùng để thử"),
    ("Tập huấn luyện, seed 0, chia phân tầng", int((df.split_s0 == "train").sum()), 1, "60%"),
    ("Tập validation, seed 0, chia phân tầng", int((df.split_s0 == "valid").sum()), 1, "20%, chỉ dùng để đặt ngưỡng 1% báo động giả"),
    ("Tập kiểm thử, seed 0, chia phân tầng", int((df.split_s0 == "test").sum()), 1, "20%"),
], columns=["buoc", "so_dong", "so_cot", "ghi_chu"])
row_counts.insert(0, "STT", range(1, len(row_counts) + 1))
row_counts.to_csv(OUT + "table_row_counts.csv", index=False)
print(row_counts.to_string(index=False))

 STT                                           buoc  so_dong  so_cot                                            ghi_chu
   1      Tệp gốc PhiUSIIL_Phishing_URL_Dataset.csv   235795      56 Dữ liệu vào (cột phishing do nhóm thêm không tính)
   2 Dòng bị xoá vì ô trống hoặc giá trị ngoài miền        0      56                              Không có dòng nào lỗi
   3                       Dòng bị xoá vì URL trùng      425      56  Giữ lần xuất hiện đầu; cả 425 dòng đều là lừa đảo
   4                          Bảng sau khi bỏ trùng   235370      56                    134.850 hợp lệ, 100.520 lừa đảo
   5                Địa chỉ là trang chủ (nhóm khó)   175363       1         Toàn bộ lớp hợp lệ và một phần lớp lừa đảo
   6                Địa chỉ có TLD là com, org, net   138220       1         Phần huấn luyện của phép thử TLD chưa thấy
   7                            Địa chỉ có TLD khác    97150       1                               Phần chỉ dùng để thử
   8         Tập huấn luyện, seed 0, chi

**Nhận xét:** dữ liệu vào có 235.795 dòng. Không dòng nào bị xoá vì ô trống hay giá trị sai miền. 425 dòng bị xoá vì URL trùng, còn 235.370 dòng, gấp gần 8 lần mức 30.000 dòng môn học ưu tiên.
Nhóm khó (địa chỉ là trang chủ) có 175.363 dòng. Phép thử TLD chưa thấy có 138.220 dòng để huấn luyện và 97.150 dòng để thử. Ở seed 0, chia phân tầng cho 141.222 dòng huấn luyện, 47.074 dòng validation và 47.074 dòng kiểm thử.

## 2.15 Lưu các bảng sản phẩm
Ô dưới lưu bảng mô hình `Outputs/phishing_model_table.parquet` (định dạng parquet giữ đúng kiểu dữ liệu và đọc nhanh hơn CSV) và tệp `Outputs/feature_groups.json` ghi tên các cột của từng nhóm đặc trưng và từng cách chia, để Bước 3 tới Bước 6 đọc lại đúng một danh sách.
Cách đọc: dòng đầu in kích thước bảng đã lưu, dòng sau liệt kê các tệp trong `Outputs`.

In [16]:
SPLITS = [f"{k}_s{s}" for s in SEEDS for k in ["split", "gsplit"]]
KEEP = ["row_id", "URL", "Domain", "reg_domain", "TLD", "tld_clean", "tld_type", "country", "phishing", "starts_https_www", "has_path", "homepage_only",
        "source_prefix", "url_length_true", "tld_seen"] + SHORTCUT + ADDRESS + PAGE + PREFIX_BLIND + SPLITS
model_table = df[KEEP]
model_table.to_parquet(OUT + "phishing_model_table.parquet", index=False)
json.dump({"shortcut": SHORTCUT, "address": ADDRESS, "page": PAGE, "prefix_blind": PREFIX_BLIND, "seeds": SEEDS, "seen_tlds": SEEN_TLDS,
           "splits": SPLITS, "target": "phishing"}, open(OUT + "feature_groups.json", "w"), indent=2, ensure_ascii=False)
check = pd.read_parquet(OUT + "phishing_model_table.parquet")
assert check.shape == model_table.shape
print("Đã lưu phishing_model_table.parquet", model_table.shape, "| feature_groups.json | và các bảng table_*.csv trong", os.path.abspath(OUT))
print(sorted(f for f in os.listdir(OUT) if f.endswith((".csv", ".parquet", ".json"))))

Đã lưu phishing_model_table.parquet (235370, 89) | feature_groups.json | và các bảng table_*.csv trong c:\Users\TGDD\Desktop\PhishingURL_Project\Outputs
['feature_groups.json', 'phishing_model_table.parquet', 'table_consistency_checks.csv', 'table_data_dictionary.csv', 'table_feature_by_class.csv', 'table_feature_statistics.csv', 'table_row_counts.csv', 'table_splits.csv', 'table_value_ranges.csv', 'tld_country_codes.csv']


**Tóm tắt Bước 2.** Bảng mô hình có 235.370 dòng x 89 cột: khoá và cột mô tả (địa chỉ, tên miền, TLD, loại TLD, tên nước), nhãn `phishing`, ba cột của quy tắc một dòng và nhóm khó, 64 đặc trưng trong bốn nhóm và 10 cột chia.
Bước này phát hiện thêm hai điều chưa có trong đề cương: cột `URLLength` và `NoOfLettersInURL` được tính theo hai cách khác nhau giữa hai lớp, và 27,39% địa chỉ kiểm thử của cách chia ngẫu nhiên có tên miền đã gặp lúc huấn luyện. Phép tự kiểm ở mục 2.12 cho thấy bộ đặc trưng không thấy tiền tố thật sự không phụ thuộc vào tiền tố. Trên nhóm trang chủ, cột `IsHTTPS` của tác giả vẫn có tương quan 0,963 với nhãn, trong khi đặc trưng không thấy tiền tố mạnh nhất chỉ có 0,299 (mục 2.13).
Bước 3 dùng SQL để hỏi các câu mô tả trên chính bảng này.